# Intelligent Cyberbullying Detection Framework
## Module: Data Preprocessing & BERT Tokenization Preparation (Person 1)

### Overview
This notebook details the **BERT-aware NLP preprocessing**, **tokenization analysis with `bert-base-uncased`**, **stratified data splitting**, and **class imbalance handling**.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('../src'))
from data_cleaning import clean_text_bert, DataCleaner
from bert_tokenization import BertTokenizationPipeline, CyberbullyingBERTDataset
from dataset_splitting import DatasetSplitter

sns.set_theme(style='whitegrid')
print('Preprocessing and BERT pipeline ready!')

## 1. BERT-Aware Contextual Text Cleaning
BERT relies heavily on bidirectional contextual attention. Therefore:
- **NO** aggressive stopword removal (pronouns and negations like *not, no, never* are crucial for insults/threats).
- **NO** stemming or lemmatization (destroys inflected English grammar).
- **Emoji Demojization**: Converts emojis (😡, 🔪) to text descriptions (`enraged face`, `kitchen knife`) to avoid `[UNK]` tokens while preserving high-intensity emotion.

In [ ]:
test_samples = [
    'RT @john_doe: You are sooooo dumb and ugly! 😡🖕 #loser https://t.co/abc1234',
    'I will find where you live and kill you! 🔪💀',
    'Dear Jesus &amp; Mary, why is #MKR so white? &quot;crapilicious&quot;',
    'I love everyone! Have a blessed day &lt;3 ❤️'
]

for s in test_samples:
    print(f'RAW  : {s}')
    print(f'CLEAN: {clean_text_bert(s)}')
    print('-' * 70)

## 2. Integrated Dataset Inspection
Load the preprocessed, deduplicated master corpus generated by `dataset_integration.py`.

In [ ]:
df_clean = pd.read_csv('../datasets/processed/cyberbullying_integrated_cleaned.csv')
print(f'Cleaned integrated dataset size: {len(df_clean):,} records')
print('\nClass distribution:')
print(df_clean['label_name'].value_counts())
df_clean[['cleaned_text', 'label_id', 'label_name', 'original_dataset']].head(5)

## 3. BERT Tokenization Analysis (`bert-base-uncased`)
We analyze the WordPiece token length distribution across the corpus to select an optimal `max_length`.

In [ ]:
tok_pipeline = BertTokenizationPipeline()
stats = tok_pipeline.analyze_sequence_lengths(df_clean['cleaned_text'].tolist(), sample_size=15000)

for k, v in stats.items():
    print(f'{k:<25}: {v}')

### 3.1 Tokenization Demonstration Across Categories
Examine how `BertTokenizer` encodes input IDs, attention masks, and special tokens (`[CLS]`, `[SEP]`, `[PAD]`).

In [ ]:
demos = tok_pipeline.demonstrate_tokenization()

## 4. Stratified Dataset Splitting & Class Imbalance Analysis
Dataset partitioned into **70% Train**, **15% Validation**, and **15% Testing**.
Calculates balanced class weights for `nn.CrossEntropyLoss(weight=class_weights)` to handle the underrepresented `Threat` class without pruning majority data.

In [ ]:
train_df = pd.read_csv('../datasets/processed/train.csv')
val_df = pd.read_csv('../datasets/processed/val.csv')
test_df = pd.read_csv('../datasets/processed/test.csv')

print(f'Train set: {len(train_df):,} samples ({len(train_df)/len(df_clean)*100:.1f}%)')
print(f'Val set  : {len(val_df):,} samples ({len(val_df)/len(df_clean)*100:.1f}%)')
print(f'Test set : {len(test_df):,} samples ({len(test_df)/len(df_clean)*100:.1f}%)')

with open('../outputs/class_weights.json') as f:
    weights_data = json.load(f)

print('\nBalanced Class Weights calculated for Model Loss:')
for cat, weight in weights_data['balanced_class_weights'].items():
    print(f'  {cat:<20}: {weight:.4f}x')

## 5. PyTorch Dataset & DataLoader Verification
Verify that `CyberbullyingBERTDataset` produces valid torch tensor batches for BERT fine-tuning.

In [ ]:
import torch
from torch.utils.data import DataLoader

ds_train = CyberbullyingBERTDataset(
    texts=train_df['cleaned_text'].head(100),
    labels=train_df['label_id'].head(100),
    tokenizer=tok_pipeline.tokenizer,
    max_length=128
)

loader = DataLoader(ds_train, batch_size=16, shuffle=True)
sample_batch = next(iter(loader))

print('PyTorch Batch Tensors successfully constructed:')
print('  input_ids shape     :', sample_batch['input_ids'].shape)
print('  attention_mask shape:', sample_batch['attention_mask'].shape)
print('  token_type_ids shape:', sample_batch['token_type_ids'].shape)
print('  labels shape        :', sample_batch['labels'].shape)
print('\nReady for BERT model training by Person 2!')